# Conexión Mongo DB

In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['DataTraders_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención


# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "DataTraders"
CATEGORIA_ACTUAL = "Mercado y Finanzas"

# --- CONEXIÓN A MONGODB (como en el código anterior) ---
try:
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXION EXITOSA: Listo para insertar datos de", CATEGORIA_ACTUAL)
except Exception as e:
    print("ERROR DE CONEXION:", e)

# --- CONFIGURACIÓN SELENIUM ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

limite_paginas = 3

try:
    driver.get("http://books.toscrape.com/")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        libros = driver.find_elements(By.CLASS_NAME, "product_pod")
        
        for libro in libros:
            try:
                # --- CAPTURA ---
                titulo = libro.find_element(By.TAG_NAME, "h3")\
                              .find_element(By.TAG_NAME, "a")\
                              .get_attribute("title")

                precio = libro.find_element(By.CLASS_NAME, "price_color").text

                # --- FORMATO UNIFICADO (como tu primer código) ---
                registro = {
                    "item": titulo,
                    "valor": float(
                        precio
                        .replace("£", "")
                        .replace(" ", "")
                        .replace(",", "")
                        .strip()
                    ),
                    "categoria": CATEGORIA_ACTUAL,
                    "grupo": NOMBRE_GRUPO,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                }

                # --- INSERCIÓN DIRECTA ---
                coleccion.insert_one(registro)

            except Exception as e:
                print("Error procesando libro:", e)

        # --- PAGINACIÓN ---
        try:
            driver.find_element(By.CSS_SELECTOR, "li.next a").click()
        except Exception:
            print("Fin del scraping.")
            break

    print("PROCESO FINALIZADO: Datos enviados a MongoDB.")

except Exception as e:
    print("ERROR GENERAL:", e)

finally:
    driver.quit()

CONEXION EXITOSA: Listo para insertar datos de Mercado y Finanzas
--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
PROCESO FINALIZADO: Datos enviados a MongoDB.
